# cv-homography-degradation — main_colab

Single Colab notebook for:
- cloning/updating the repo
- installing requirements
- downloading HPatches into the current Colab session
- writing a Colab runtime config
- sanity-checking the dataset loader
- running ORB, XFeat, or Proposed experiments
- summarizing and inspecting outputs

This notebook uses **local session storage** for HPatches, not Google Drive.

In [ ]:
# --- User settings ---
REPO_URL = "https://github.com/khalayli/cv-homography-degradation"
REPO_DIR = "/content/cv-homography-degradation"

DATASET_SAVE_DIR = "/content/data/hpatches"
DATASET_NAME = "hpatches-sequences-release"
HPATCHES_URL = "https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip"

FORCE_REDOWNLOAD = False
FORCE_UNZIP = False

LIMIT_PAIRS = 10          # set to None for a fuller run
MATCHER_NAME = "orb"      # choose: "orb", "xfeat", or "proposed"

RUNTIME_CONFIG_PATH = "configs/colab_runtime.yaml"

print(f"[settings] REPO_URL={REPO_URL}")
print(f"[settings] REPO_DIR={REPO_DIR}")
print(f"[settings] DATASET_SAVE_DIR={DATASET_SAVE_DIR}")
print(f"[settings] DATASET_NAME={DATASET_NAME}")
print(f"[settings] LIMIT_PAIRS={LIMIT_PAIRS}")
print(f"[settings] MATCHER_NAME={MATCHER_NAME}")
print(f"[settings] RUNTIME_CONFIG_PATH={RUNTIME_CONFIG_PATH}")

In [ ]:
# --- Clone or update the repo, then install requirements ---
import os
import subprocess
import sys


def run_cmd(cmd, cwd=None):
    print(f"[run_cmd] cwd={cwd} cmd={' '.join(cmd)}")
    completed = subprocess.run(cmd, cwd=cwd, check=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    return completed


if not os.path.exists(REPO_DIR):
    print("[repo] Repo not found locally. Cloning now...")
    run_cmd(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("[repo] Repo already exists. Pulling latest changes...")
    run_cmd(["git", "pull"], cwd=REPO_DIR)

print("[repo] Installing requirements...")
run_cmd([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO_DIR)

os.chdir(REPO_DIR)
print(f"[repo] Working directory set to: {os.getcwd()}")

In [ ]:
# --- Ensure HPatches exists in the current Colab session ---
import os
import subprocess
from pathlib import Path


def _looks_like_hpatches_root(extracted_path):
    print(f"[_looks_like_hpatches_root] Checking extracted_path={extracted_path}")
    root = Path(extracted_path)

    if not root.exists() or not root.is_dir():
        print("[_looks_like_hpatches_root] Root does not exist or is not a directory.")
        return False

    scene_dirs = [p for p in root.iterdir() if p.is_dir()]
    print(f"[_looks_like_hpatches_root] Found {len(scene_dirs)} scene directories.")

    if len(scene_dirs) == 0:
        print("[_looks_like_hpatches_root] No scene directories found.")
        return False

    sample_scene = scene_dirs[0]
    print(f"[_looks_like_hpatches_root] Sample scene={sample_scene}")

    sample_img_1 = sample_scene / "1.ppm"
    sample_img_2 = sample_scene / "2.ppm"
    sample_h = sample_scene / "H_1_2"

    ok = sample_img_1.exists() and sample_img_2.exists() and sample_h.exists()
    print(
        f"[_looks_like_hpatches_root] "
        f"1.ppm={sample_img_1.exists()}, "
        f"2.ppm={sample_img_2.exists()}, "
        f"H_1_2={sample_h.exists()}"
    )
    print(f"[_looks_like_hpatches_root] ok={ok}")
    return ok


def ensure_hpatches_local(
    save_dir="/content/data/hpatches",
    dataset_name="hpatches-sequences-release",
    url="https://huggingface.co/datasets/vbalnt/hpatches/resolve/main/hpatches-sequences-release.zip",
    force_redownload=False,
    force_unzip=False,
):
    print("[ensure_hpatches_local] Starting dataset check...")
    print(f"[ensure_hpatches_local] save_dir={save_dir}")
    os.makedirs(save_dir, exist_ok=True)

    zip_path = os.path.join(save_dir, f"{dataset_name}.zip")
    extracted_path = os.path.join(save_dir, dataset_name)

    print(f"[ensure_hpatches_local] zip_path={zip_path}")
    print(f"[ensure_hpatches_local] extracted_path={extracted_path}")

    if force_redownload:
        print("[ensure_hpatches_local] force_redownload=True")
        if os.path.exists(zip_path):
            os.remove(zip_path)
            print("[ensure_hpatches_local] Existing zip removed.")
        if os.path.exists(extracted_path):
            subprocess.run(["rm", "-rf", extracted_path], check=True)
            print("[ensure_hpatches_local] Existing extracted folder removed.")

    if not force_unzip and _looks_like_hpatches_root(extracted_path):
        print("[ensure_hpatches_local] Valid extracted dataset already exists. Skipping download/unzip.")
        return extracted_path

    if not os.path.exists(zip_path):
        print("[ensure_hpatches_local] Zip not found. Downloading...")
        subprocess.run(["wget", "-c", url, "-O", zip_path], check=True)
        print("[ensure_hpatches_local] Download complete.")
    else:
        print("[ensure_hpatches_local] Zip already exists in current session.")

    if force_unzip and os.path.exists(extracted_path):
        print("[ensure_hpatches_local] force_unzip=True, removing existing extracted folder.")
        subprocess.run(["rm", "-rf", extracted_path], check=True)
        print("[ensure_hpatches_local] Existing extracted folder removed for fresh unzip.")

    if not _looks_like_hpatches_root(extracted_path):
        print("[ensure_hpatches_local] Valid extracted dataset not found. Unzipping dataset...")
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", save_dir], check=True)
        print("[ensure_hpatches_local] Unzip complete.")
    else:
        print("[ensure_hpatches_local] Extracted folder already valid. Skipping unzip.")

    if not _looks_like_hpatches_root(extracted_path):
        raise RuntimeError(
            f"[ensure_hpatches_local] Dataset extraction completed, but structure still looks invalid: {extracted_path}"
        )

    print("[ensure_hpatches_local] Final directory listing:")
    subprocess.run(["ls", "-lah", save_dir], check=True)

    print(f"[ensure_hpatches_local] Dataset ready at: {extracted_path}")
    return extracted_path


dataset_root = ensure_hpatches_local(
    save_dir=DATASET_SAVE_DIR,
    dataset_name=DATASET_NAME,
    url=HPATCHES_URL,
    force_redownload=FORCE_REDOWNLOAD,
    force_unzip=FORCE_UNZIP,
)
print(f"[main] dataset_root={dataset_root}")

In [ ]:
# --- Add repo to Python path and write a Colab-specific runtime config ---
import os
import sys
from pathlib import Path
import yaml

repo_path = Path(REPO_DIR).resolve()
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
    print(f"[pythonpath] Added repo path: {repo_path}")
else:
    print(f"[pythonpath] Repo path already present: {repo_path}")

base_cfg_path = repo_path / "configs" / "default.yaml"
runtime_cfg_path = repo_path / RUNTIME_CONFIG_PATH
print(f"[config] base_cfg_path={base_cfg_path}")
print(f"[config] runtime_cfg_path={runtime_cfg_path}")

with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["dataset"]["hpatches_root"] = dataset_root
cfg["run"]["out_dir"] = "results/colab_run"
cfg["run"]["name"] = f"colab_{MATCHER_NAME}"

cfg["method"]["name"] = MATCHER_NAME

if MATCHER_NAME == "proposed":
    if "proposed" not in cfg["method"]:
        cfg["method"]["proposed"] = {}

    cfg["method"]["proposed"]["enabled"] = True
    cfg["method"]["proposed"].setdefault("adaptive_thresholds", True)
    cfg["method"]["proposed"].setdefault("fallback_enabled", True)
    cfg["method"]["proposed"].setdefault("primary_matcher", "xfeat")
    cfg["method"]["proposed"].setdefault("fallback_matcher", "orb")
    cfg["method"]["proposed"].setdefault("min_matches_before_fallback", 30)

runtime_cfg_path.parent.mkdir(parents=True, exist_ok=True)

with open(runtime_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"[config] Wrote runtime config to: {runtime_cfg_path}")
print("[config] Runtime config contents:")
with open(runtime_cfg_path, "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# --- Sanity check the HPatches loader ---
from src.data.hpatches import HPatchesDataset

print("[sanity] Building HPatchesDataset...")
dataset = HPatchesDataset(root=dataset_root, pairs_per_scene=5)

if hasattr(dataset, "describe"):
    summary = dataset.describe()
    print(f"[sanity] summary={summary}")
elif hasattr(dataset, "summarize"):
    summary = dataset.summarize()
    print(f"[sanity] summary={summary}")
else:
    print("[sanity] Dataset has no describe/summarize method")

if hasattr(dataset, "pairs"):
    print(f"[sanity] num_pairs={len(dataset.pairs)}")
elif hasattr(dataset, "samples"):
    print(f"[sanity] num_pairs={len(dataset.samples)}")

example = dataset[0]

if hasattr(example, "pair"):
    print(f"[sanity] scene_name={example.pair.scene_name}")
    print(f"[sanity] split={example.pair.split}")
    print(f"[sanity] image0_shape={example.image0.shape}")
    print(f"[sanity] image1_shape={example.image1.shape}")
    print(f"[sanity] H_gt_shape={example.homography_gt.shape}")
else:
    print(f"[sanity] scene_name={example.get('scene_name', example.get('scene'))}")
    print(f"[sanity] split={example.get('split')}")
    print(f"[sanity] image0_shape={example.get('image0', example.get('image_src')).shape}")
    print(f"[sanity] image1_shape={example.get('image1', example.get('image_tgt')).shape}")
    print(f"[sanity] H_gt_shape={example.get('homography_gt', example.get('H_gt')).shape}")

In [ ]:
# --- Visualize one pair quickly ---
import matplotlib.pyplot as plt
import cv2

if hasattr(example, "pair"):
    img0 = example.image0
    img1 = example.image1
    title_scene = example.pair.scene_name
    title_pair = example.pair.pair_index
else:
    img0 = example.get("image0", example.get("image_src"))
    img1 = example.get("image1", example.get("image_tgt"))
    title_scene = example.get("scene_name", example.get("scene"))
    title_pair = example.get("pair_index", example.get("pair"))

img0_rgb = cv2.cvtColor(img0, cv2.COLOR_BGR2RGB)
img1_rgb = cv2.cvtColor(img1, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.imshow(img0_rgb)
plt.title(f"source: {title_scene}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img1_rgb)
plt.title(f"target pair: {title_pair}")
plt.axis("off")
plt.show()

In [ ]:
# --- Run experiment ---
import subprocess
import sys

cmd = [sys.executable, "scripts/run_experiment.py", "--config", RUNTIME_CONFIG_PATH]
if LIMIT_PAIRS is not None:
    cmd += ["--limit-pairs", str(LIMIT_PAIRS)]

print(f"[experiment] cmd={' '.join(cmd)}")
subprocess.run(cmd, check=True)
print("[experiment] Done")

In [ ]:
# --- Summarize results ---
import subprocess
import sys

metrics_csv = "results/colab_run/metrics.csv"
cmd = [sys.executable, "scripts/summarize_results.py", "--input", metrics_csv]
print(f"[summary] cmd={' '.join(cmd)}")
subprocess.run(cmd, check=True)
print("[summary] Done")

In [ ]:
# --- Inspect outputs ---
import os
import pandas as pd
from IPython.display import display

metrics_csv = "results/colab_run/metrics.csv"
summary_json = "results/colab_run/summary.json"
summary_csv = "results/colab_run/summary.csv"
summary_aggregated_json = "results/colab_run/summary_aggregated.json"

for path in [metrics_csv, summary_json, summary_csv, summary_aggregated_json]:
    print(f"[outputs] {path} exists={os.path.exists(path)}")

metrics_df = pd.read_csv(metrics_csv)
print(f"[outputs] metrics_df_shape={metrics_df.shape}")
display(metrics_df.head())

if os.path.exists(summary_csv):
    summary_df = pd.read_csv(summary_csv)
    print(f"[outputs] summary_df_shape={summary_df.shape}")
    display(summary_df.head())

In [ ]:
# --- Optional: switch matcher and rerun ---
# Set MATCHER_NAME in the settings cell to one of:
# - "orb"
# - "xfeat"
# - "proposed"
#
# Then rerun these cells:
# 1) the runtime-config cell
# 2) the experiment cell
# 3) the summary cell
# 4) the outputs cell
print("[note] Change MATCHER_NAME in the settings cell, then rerun config + experiment + summary + outputs.")